In [1]:
# HTP printability index over W–Ti–V–Zr ternaries
# ------------------------------------------------
# Files required in the same folder (or set paths below):
#   - Thermophysical_Properties_W_V_Ti_Zr.xlsx   (from previous step)
#   - WTiV.xlsx, WTiZr.xlsx, WVZr.xlsx            (composition lists), now with "TL [K]" and "TS [K]"
#
# Output:
#   - HTP_Printability_Results.xlsx  (one sheet per system)

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------- User parameters ---------------------
# Paths (edit if your files are elsewhere)
PROP_XLSX = Path("Thermophysical_Properties_W_V_Ti_Zr.xlsx")
COMP_FILES = {
    "WTiV": Path("WTiV.xlsx"),
    "WTiZr": Path("WTiZr.xlsx"),
    "WVZr": Path("WVZr.xlsx"),
}
OUTPUT_XLSX = Path("HTP_Printability_Results.xlsx")

# Process/physics parameters
Ta = 298.0               # ambient / substrate temperature [K]
a_drop = 100e-6          # initial droplet radius "a" [m]
DELTA_T_FREEZE = 100.0   # fallback freezing range [K] if TS not provided

# Small numerical guards
EPS = 1e-9

# --------------------- Helpers ---------------------
ATOMIC_WEIGHT = {  # g/mol
    "W": 183.84, "Ti": 47.867, "V": 50.9415, "Zr": 91.224
}

# Columns expected in the properties sheet (exact names from our earlier file)
PROP_COLS = {
    "T0": "T0_liquidus (K)",
    "Tf": "Tf_solidus (K)",
    "k_liq": "k_liquidus (W/mK)",
    "ka_amb": "ka_ambient (W/mK)",
    "cp_liq": "cp_liquidus (J/kgK)",
    "rho_liq": "rho_liquidus (kg/m3)",
    "L_mass": "L (J/kg)",
    "sigma": "sigma_liquidus (N/m)",
}

ELEMENTS = ["W", "Ti", "V", "Zr"]

def load_element_props(xlsx_path: Path):
    dfp = pd.read_excel(xlsx_path)
    dfp = dfp.set_index("Element")
    props = {}
    for el in ELEMENTS:
        if el in dfp.index:
            p = dfp.loc[el]
            props[el] = {
                "T0":    float(p[PROP_COLS["T0"]]),
                "Tf":    float(p[PROP_COLS["Tf"]]),
                "k_liq": float(p[PROP_COLS["k_liq"]]),
                "ka":    float(p[PROP_COLS["ka_amb"]]),
                "cp":    float(p[PROP_COLS["cp_liq"]]),
                "rho":   float(p[PROP_COLS["rho_liq"]]),
                "L":     float(p[PROP_COLS["L_mass"]]),  # per mass
                "sigma": float(p[PROP_COLS["sigma"]]),
            }
    return props

def normalize_mole_fractions(row):
    """Accept mole fractions or atomic %, auto-normalize to sum=1 for present elements."""
    x = np.array([row.get(el, 0.0) for el in ELEMENTS], dtype=float)
    total = np.nansum(x)
    if total <= 0:
        return dict(zip(ELEMENTS, np.zeros_like(x)))
    if total > 1.5:  # likely percentages
        x = x / total
    else:
        x = x / (total + EPS)
    return dict(zip(ELEMENTS, x))

def mole_to_mass_fractions(x_mole):
    moles = np.array([x_mole.get(el, 0.0) for el in ELEMENTS], dtype=float)
    masses = moles * np.array([ATOMIC_WEIGHT[el] for el in ELEMENTS])
    total_mass = np.nansum(masses)
    if total_mass <= 0:
        return dict(zip(ELEMENTS, np.zeros_like(masses)))
    w = masses / total_mass
    return dict(zip(ELEMENTS, w))

def mix_linear_mass_weighted(w_mass, key, props):
    return float(sum(w_mass[el] * props[el][key] for el in ELEMENTS if el in props))

def mix_density(w_mass, props):
    denom = sum(w_mass[el] / props[el]["rho"] for el in ELEMENTS if el in props and props[el]["rho"] > 0)
    return 1.0 / max(denom, EPS)

def mix_T_liquidus(x_mole, props):
    """Approximate liquidus by mole-fraction average of elemental Tm (fallback only)."""
    return float(sum(x_mole[el] * props[el]["T0"] for el in ELEMENTS if el in props))

def mix_T_solidus(T0_liq):
    """Fallback solidus by subtracting a tunable freezing range."""
    return max(T0_liq - DELTA_T_FREEZE, 1.0)

def compute_tau_terms(a, Ta, k_liq, ka_amb, L_mass, cp_liq, T0, Tf, rho_liq, sigma):
    dT0 = max(T0 - Ta, 1e-6)
    dTf = max(Tf - Ta, 1e-6)
    ln_arg = max(dT0 / dTf, 1.0)  # ensures non-negative ln
    pref = (a * k_liq) / (3.0 * max(ka_amb, EPS))
    tau1 = pref * np.log(ln_arg)
    L_over_C = L_mass / max(cp_liq, EPS)  # density cancels
    tau2 = pref * (1.0 + (ka_amb / max(2.0 * k_liq, EPS))) * (L_over_C / max((Tf - Ta), 1e-6))
    tau_solid = 2.0 * (tau1 + tau2)
    tau_spread = np.sqrt((rho_liq * (a**3)) / max(sigma, EPS))
    PI = tau_solid / max(tau_spread, EPS)
    return tau1, tau2, tau_solid, tau_spread, PI

def row_TL_TS(row):
    """Safely fetch TL/TS from row if available and finite; else return (np.nan, np.nan)."""
    TL = row.get("TL [K]", np.nan)
    TS = row.get("TS [K]", np.nan)
    try:
        TL = float(TL)
    except Exception:
        TL = np.nan
    try:
        TS = float(TS)
    except Exception:
        TS = np.nan
    # Reject non-physical ordering
    if not np.isfinite(TL) or not np.isfinite(TS) or TL < TS:
        return np.nan, np.nan
    return TL, TS

def process_system(name, comp_path, props):
    dfc = pd.read_excel(comp_path)

    # Ensure element columns exist (fill missing with zeros)
    for el in ELEMENTS:
        if el not in dfc.columns:
            dfc[el] = 0.0

    results = []
    for _, row in dfc.iterrows():
        # 1) normalize mole fractions (or at.%)
        x = normalize_mole_fractions(row)
        # 2) mass fractions
        w = mole_to_mass_fractions(x)

        # 3) mix properties at liquidus (k, ka, cp, rho, L, sigma)
        k_liq  = mix_linear_mass_weighted(w, "k_liq", props)
        ka_amb = mix_linear_mass_weighted(w, "ka",    props)
        cp_liq = mix_linear_mass_weighted(w, "cp",    props)
        rho_liq= mix_density(w, props)
        L_mass = mix_linear_mass_weighted(w, "L",     props)   # J/kg
        sigma  = mix_linear_mass_weighted(w, "sigma", props)

        # 4) Use per-row TL/TS if provided; otherwise fall back to old estimate
        TL_row, TS_row = row_TL_TS(row)
        if np.isfinite(TL_row) and np.isfinite(TS_row):
            T0 = TL_row
            Tf = TS_row
            source_T = "row"
        else:
            T0 = mix_T_liquidus(x, props)
            Tf = mix_T_solidus(T0)
            source_T = "fallback"

        # 5) derived
        alpha  = k_liq / max(rho_liq * cp_liq, EPS)

        # 6) timescales
        tau1, tau2, tau_solid, tau_spread, PI = compute_tau_terms(
            a=a_drop, Ta=Ta, k_liq=k_liq, ka_amb=ka_amb, L_mass=L_mass,
            cp_liq=cp_liq, T0=T0, Tf=Tf, rho_liq=rho_liq, sigma=sigma
        )

        out = {
            "W_mole": x["W"], "Ti_mole": x["Ti"], "V_mole": x["V"], "Zr_mole": x["Zr"],
            "W_mass": w["W"], "Ti_mass": w["Ti"], "V_mass": w["V"], "Zr_mass": w["Zr"],
            "T0_liquidus[K] (used)": T0,
            "Tf_solidus[K] (used)": Tf,
            "k_liq[W/mK]": k_liq, "ka_ambient[W/mK]": ka_amb,
            "cp_liq[J/kgK]": cp_liq, "rho_liq[kg/m3]": rho_liq,
            "alpha_liq[m2/s]": alpha,
            "L_mass[J/kg]": L_mass, "sigma_liq[N/m]": sigma,
            "tau1[s]": tau1, "tau2[s]": tau2, "tau_solid[s]": tau_solid,
            "tau_spread[s]": tau_spread, "PI = tau_solid/tau_spread": PI,
            "T_source": source_T,  # for traceability: "row" or "fallback"
        }
        results.append(out)

    out_df = pd.DataFrame(results)
    # keep original columns too (for traceability)
    for col in dfc.columns:
        if col not in out_df.columns:
            out_df[col] = dfc[col]

    preferred = [
        "W_mole","Ti_mole","V_mole","Zr_mole",
        "W_mass","Ti_mass","V_mass","Zr_mass",
        "T0_liquidus[K] (used)","Tf_solidus[K] (used)",
        "k_liq[W/mK]","ka_ambient[W/mK]",
        "cp_liq[J/kgK]","rho_liq[kg/m3]","alpha_liq[m2/s]",
        "L_mass[J/kg]","sigma_liq[N/m]",
        "tau1[s]","tau2[s]","tau_solid[s]","tau_spread[s]",
        "PI = tau_solid/tau_spread",
        "T_source",
        "TL [K]","TS [K]",  # your inputs, preserved
    ]
    out_df = out_df[[c for c in preferred if c in out_df.columns] +
                    [c for c in out_df.columns if c not in preferred]]
    return out_df

# --------------------- Run ---------------------
props = load_element_props(PROP_XLSX)

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    for sys_name, comp_path in COMP_FILES.items():
        sheet_df = process_system(sys_name, comp_path, props)
        sheet_df.to_excel(writer, sheet_name=sys_name, index=False)

print(f"Done. Wrote results to: {OUTPUT_XLSX.resolve()}")


Done. Wrote results to: /Users/user/Library/CloudStorage/OneDrive-Personal/ISU First semester/_Research - COMPASS Lab/Printability map/csc v2/SS_index_v2/HTP_Printability_Results.xlsx


In [3]:
import os

current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

Current working directory: /Users/user/Library/CloudStorage/OneDrive-Personal/ISU First semester/_Research - COMPASS Lab/Printability map/csc v2/SS_index_v2
